# 2. GitHub Actions OIDC con Vault

Este notebook configura autenticación OIDC de GitHub Actions contra Vault usando `vault` CLI para el lado de Vault y `gh` CLI para el lado de GitHub.

In [1]:
%env WORKDIR = '/tmp/vault'
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

env: WORKDIR='/tmp/vault'


In [2]:
WORKDIR = '/tmp/vault'
REPO_FULL_NAME = 'jm-merchan/Vault_Use_Cases_Example_202607'
REPO_OWNER, REPO_NAME = REPO_FULL_NAME.split('/', 1)
GHA_BRANCH = 'main'
VAULT_JWT_PATH = 'github'
VAULT_POLICY_NAME = f'gha-{REPO_NAME}'
VAULT_ROLE_NAME = f'{VAULT_POLICY_NAME}-{GHA_BRANCH}'

os.environ.update({
    'WORKDIR': WORKDIR,
    'REPO_FULL_NAME': REPO_FULL_NAME,
    'REPO_OWNER': REPO_OWNER,
    'REPO_NAME': REPO_NAME,
    'GHA_BRANCH': GHA_BRANCH,
    'VAULT_JWT_PATH': VAULT_JWT_PATH,
    'VAULT_POLICY_NAME': VAULT_POLICY_NAME,
    'VAULT_ROLE_NAME': VAULT_ROLE_NAME,
})
os.environ.pop('VAULT_SKIP_VERIFY', None)
os.makedirs(f'{WORKDIR}/gha', exist_ok=True)

## Policy de Vault para GitHub Actions

La policy de ejemplo permite leer secretos en `secret/data/gha/*` (KV v2). Ajusta paths/capabilities según tu caso.

In [3]:
%%bash
set -euo pipefail

vault status
if ! vault secrets list -format=json | jq -e 'has("secret/")' >/dev/null; then
  vault secrets enable -path=secret -version=2 kv
fi

cat > ${WORKDIR}/gha/${VAULT_POLICY_NAME}.hcl <<EOF
path "secret/data/gha/*" {
  capabilities = ["read"]
}

path "secret/metadata/gha/*" {
  capabilities = ["read", "list"]
}
EOF

vault policy write ${VAULT_POLICY_NAME} ${WORKDIR}/gha/${VAULT_POLICY_NAME}.hcl
vault policy read ${VAULT_POLICY_NAME}
vault kv put secret/gha/demo api_key="$(openssl rand -hex 16)"

Key                      Value
---                      -----
Seal Type                awskms
Recovery Seal Type       shamir
Initialized              true
Sealed                   false
Total Recovery Shares    1
Threshold                1
Version                  2.1.1+ent
Build Date               2026-09-15T21:39:40Z
Storage Type             raft
Cluster Name             vault-cluster-ceed5b26
Cluster ID               bf46dd82-ead9-248c-5df1-512604c3b7d6
Removed From Cluster     false
HA Enabled               true
HA Cluster               https://vault-0.vault-internal:8201
HA Mode                  active
Active Since             2026-09-23T10:36:41.707897345Z
Raft Committed Index     235
Raft Applied Index       235
Last WAL                 88
Success! Enabled the kv secrets engine at: secret/


Policy name was converted from "gha-Vault_Use_Cases_Example_202607" to "gha-vault_use_cases_example_202607"


Success! Uploaded policy: gha-vault_use_cases_example_202607
path "secret/data/gha/*" {
  capabilities = ["read"]
}

path "secret/metadata/gha/*" {
  capabilities = ["read", "list"]
}
==== Secret Path ====
secret/data/gha/demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-09-23T10:37:40.414494588Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1


In [4]:
%%bash
set -euo pipefail
if ! vault auth list -format=json | jq -e --arg path "${VAULT_JWT_PATH}/" 'has($path)' >/dev/null; then
  vault auth enable -path=${VAULT_JWT_PATH} jwt
fi

vault write auth/${VAULT_JWT_PATH}/config \
  oidc_discovery_url="https://token.actions.githubusercontent.com" \
  bound_issuer="https://token.actions.githubusercontent.com"

vault read auth/${VAULT_JWT_PATH}/config

Success! Enabled jwt auth method at: github/
Success! Data written to: auth/github/config
Key                                     Value
---                                     -----
bound_issuer                            https://token.actions.githubusercontent.com
default_role                            n/a
jwks_ca_pem                             n/a
jwks_pairs                              []
jwks_url                                n/a
jwt_supported_algs                      []
jwt_validation_pubkeys                  []
namespace_in_state                      true
oidc_client_id                          n/a
oidc_discovery_ca_pem                   n/a
oidc_discovery_url                      https://token.actions.githubusercontent.com
oidc_response_mode                      n/a
oidc_response_types                     []
provider_config                         map[]
unsupported_critical_cert_extensions    []


In [5]:
%%bash
set -euo pipefail
cat > ${WORKDIR}/gha/role-${VAULT_ROLE_NAME}.json <<EOF
{
  "role_type": "jwt",
  "user_claim": "actor",
  "bound_audiences": "https://github.com/${REPO_OWNER}",
  "bound_claims_type": "glob",
  "bound_claims": {
    "repository": "${REPO_FULL_NAME}",
    "ref": "refs/heads/${GHA_BRANCH}"
  },
  "token_policies": "${VAULT_POLICY_NAME}",
  "token_ttl": "1h"
}
EOF

vault write auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME} @${WORKDIR}/gha/role-${VAULT_ROLE_NAME}.json
vault read auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME}

Success! Data written to: auth/github/role/gha-Vault_Use_Cases_Example_202607-main
Key                        Value
---                        -----
alias_metadata             map[]
allowed_redirect_uris      <nil>
bound_audiences            [https://github.com/jm-merchan]
bound_claims               map[ref:refs/heads/main repository:jm-merchan/Vault_Use_Cases_Example_202607]
bound_claims_type          glob
bound_subject              n/a
claim_mappings             <nil>
clock_skew_leeway          0
expiration_leeway          0
groups_claim               n/a
max_age                    0
not_before_leeway          0
oidc_scopes                <nil>
role_type                  jwt
token_bound_cidrs          []
token_explicit_max_ttl     0s
token_max_ttl              0s
token_no_default_policy    false
token_num_uses             0
token_period               0s
token_policies             [gha-Vault_Use_Cases_Example_202607]
token_ttl                  1h
token_type                 default
use

## Workflow de ejemplo en GitHub Actions

Este job solicita un token OIDC (`id-token: write`), autentica contra Vault y lee un secreto.

Requisitos en el repositorio de GitHub:
- Variable `VAULT_ADDR` (URL de Vault, por ejemplo `https://...`)
- Variable `VAULT_AUTH_PATH` (en este ejemplo `github`)
- Variable `VAULT_AUTH_ROLE` (rol creado en este notebook)

In [6]:
%%bash
set -euo pipefail

WORKFLOW_FILE=.github/workflows/vault-oidc.yml
mkdir -p .github/workflows
cat > ${WORKFLOW_FILE} <<'EOF'
name: vault-oidc
on:
  workflow_dispatch:
jobs:
  read-secret:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      id-token: write
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Import Vault Secrets
        id: vault
        uses: hashicorp/vault-action@v3
        with:
          url: ${{ vars.VAULT_ADDR }}
          tlsSkipVerify: true
          method: jwt
          path: ${{ vars.VAULT_AUTH_PATH }}
          role: ${{ vars.VAULT_AUTH_ROLE }}
          secrets: |
            secret/data/gha/demo api_key | API_KEY

      - name: Use Secret
        run: |
          test -n "${API_KEY}"
          echo "Secret loaded successfully"
EOF

gh variable set VAULT_AUTH_PATH --repo "${REPO_FULL_NAME}" --body "${VAULT_JWT_PATH}"
gh variable set VAULT_AUTH_ROLE --repo "${REPO_FULL_NAME}" --body "${VAULT_ROLE_NAME}"

if [ -n "${VAULT_ADDR:-}" ]; then
  gh variable set VAULT_ADDR --repo "${REPO_FULL_NAME}" --body "${VAULT_ADDR}"
fi

echo "Workflow generado localmente en ${WORKFLOW_FILE}"
echo "Variables GHA configuradas: VAULT_AUTH_PATH, VAULT_AUTH_ROLE${VAULT_ADDR:+, VAULT_ADDR}"

Workflow generado localmente en .github/workflows/vault-oidc.yml
Sube el workflow al repo jm-merchan/Vault_Use_Cases_Example_202607 con git push o gh api repos/.../contents si prefieres API.
Variables GHA configuradas: VAULT_AUTH_PATH, VAULT_AUTH_ROLE, VAULT_ADDR


# Upload action to github

In [7]:
%%bash
set -euo pipefail

WORKFLOW_PATH=".github/workflows/vault-oidc.yml"
test -s "${WORKFLOW_PATH}"
CONTENT="$(base64 < "${WORKFLOW_PATH}" | tr -d '\n')"

EXISTING_SHA="$(gh api "repos/${REPO_FULL_NAME}/contents/${WORKFLOW_PATH}?ref=${GHA_BRANCH}" --jq .sha 2>/dev/null || true)"

ARGS=(
  --method PUT
  -H "Accept: application/vnd.github+json"
  "repos/${REPO_FULL_NAME}/contents/${WORKFLOW_PATH}"
  -f "message=Add Vault OIDC GitHub Action"
  -f "branch=${GHA_BRANCH}"
  -f "content=${CONTENT}"
)
if [[ -n "${EXISTING_SHA}" ]]; then
  ARGS+=(-f "sha=${EXISTING_SHA}")
fi

gh api "${ARGS[@]}" --jq '.content.path + " committed " + .commit.sha'


.github/workflows/vault-oidc.yml committed 92b602e932effc8cf154d3a98ac498947526dbb3


# Run the action

In [ ]:
%%bash
set -euo pipefail

WORKFLOW_FILE="vault-oidc.yml"
PREVIOUS_RUN_ID="$(gh run list \
  --repo "${REPO_FULL_NAME}" \
  --workflow "${WORKFLOW_FILE}" \
  --branch "${GHA_BRANCH}" \
  --limit 1 \
  --json databaseId \
  --jq '.[0].databaseId // empty')"

gh workflow run "${WORKFLOW_FILE}" \
  --repo "${REPO_FULL_NAME}" \
  --ref "${GHA_BRANCH}"

RUN_ID=""
for _ in $(seq 1 30); do
  RUN_ID="$(gh run list \
    --repo "${REPO_FULL_NAME}" \
    --workflow "${WORKFLOW_FILE}" \
    --branch "${GHA_BRANCH}" \
    --event workflow_dispatch \
    --limit 1 \
    --json databaseId \
    --jq '.[0].databaseId // empty')"
  if [[ -n "${RUN_ID}" && "${RUN_ID}" != "${PREVIOUS_RUN_ID}" ]]; then
    break
  fi
  RUN_ID=""
  sleep 2
done

test -n "${RUN_ID}"
echo "Watching run ${RUN_ID}"
gh run watch "${RUN_ID}" --repo "${REPO_FULL_NAME}" --exit-status
gh run view "${RUN_ID}" --repo "${REPO_FULL_NAME}"


## CLEAN UP

Elimina únicamente el auth method, rol, política y secreto creados por este notebook. Está protegido para evitar ejecuciones accidentales.

In [ ]:
%%bash
set -euo pipefail
CONFIRM_CLEANUP=${CONFIRM_CLEANUP:-false}
if [[ "${CONFIRM_CLEANUP}" != "DELETE_GHA_OIDC_DEMO" ]]; then
  echo 'Cleanup omitido. Usa CONFIRM_CLEANUP=DELETE_GHA_OIDC_DEMO.'
  exit 0
fi
vault auth disable "${VAULT_JWT_PATH}" 2>/dev/null || true
vault policy delete "${VAULT_POLICY_NAME}" 2>/dev/null || true
vault kv metadata delete secret/gha/demo 2>/dev/null || true
rm -rf "${WORKDIR:?}/gha"
echo 'Cleanup del notebook 2 completado.'